## INIT 

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger
from datetime import datetime
from utils.pipeline_monitoring import log_pipeline_run

dataset_name = "fuel_type_manufacturer_dim"
layer = "gold"

start_time = datetime.now()

logger = get_project_logger("Gold_Dimension_Manufacturer")
logger.info("--- Starting Gold Layer: Creating dim_manufacturer ---")

## Creating a gold dimension view

In [0]:
start_time = datetime.now()

try:
    logger.info("Step 1/2: Running SQL to create dim_manufacturer view")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.dim_manufacturer AS
    WITH base AS (
      SELECT
        CAST(manufacturer_cd AS STRING) AS manufacturer_cd,
        TRIM(manufacturer_nm) AS manufacturer_nm
      FROM (
        SELECT manufacturer_cd, manufacturer_nm
        FROM transport_lakehouse.silver.silver_vehicles_two_wheeled
        UNION ALL
        SELECT manufacturer_cd, manufacturer_nm
        FROM transport_lakehouse.silver.silver_vehicles_public
        UNION ALL
        SELECT manufacturer_cd, manufacturer_nm
        FROM transport_lakehouse.silver.silver_vehicles_private
      )
    ),
    dedup AS (
      SELECT
        manufacturer_cd,
        MAX(manufacturer_nm) AS manufacturer_nm
      FROM base
      WHERE manufacturer_cd IS NOT NULL
      GROUP BY manufacturer_cd
    )
    SELECT
      ROW_NUMBER() OVER (ORDER BY manufacturer_cd) AS manufacturer_key,
      manufacturer_cd,
      manufacturer_nm
    FROM dedup
    """
    
    spark.sql(sql_query)
    logger.info("Step 1/2: View created successfully.")

    logger.info("Step 2/2: Performing Quality Check (Row Count)")
    dim_count = spark.table("transport_lakehouse.gold.dim_manufacturer").count()
    logger.info(f"Quality Check Passed: dim_manufacturer contains {dim_count} unique manufacturers.")

    end_time = datetime.now()

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="SUCCESS",
        rows_ingested=dim_count,
        error_message=None
    )

except Exception as e:
    end_time = datetime.now()
    logger.error(f"Failed to create Gold Dimension: {str(e)}")

    log_pipeline_run(
        spark=spark,
        dataset_name=dataset_name,
        layer=layer,
        run_start_time=start_time,
        run_end_time=end_time,
        status="FAILED",
        rows_ingested=0,
        error_message=str(e)
    )

    raise

logger.info("--- Gold dim_manufacturer Process Completed ---")